# Gradient Descent: Fitting by Searching

Every model we've fit so far has a fitting step: choose the numbers that make the model match the data. For linear regression a formula does it, and for naive Bayes counting does it. Logistic regression has no formula, so the computer **searches** for its best coefficients. This notebook shows how that search works.

The whole idea fits in one line: **find which way is downhill on the loss, take a step that way, and repeat.** That's gradient descent.

### By the end, you'll be able to

1. Know which models we've seen are fit by a formula, which by counting, and which by searching.
2. Describe fitting as finding the minimum of a loss curve (or surface).
3. Use the slope to say which way is downhill and how steep it is.
4. Apply the update rule by hand, and say what the minus sign and the learning rate $\eta$ do.
5. Predict what too small or too large a learning rate does, and decide when to stop searching (minimizing loss).
6. Read a contour plot as a loss surface seen from above, and read partial derivatives and the gradient from it.
7. Explain why gradient descent becomes inefficient (zigzags) sometimes, when features aren't scaled.
8. Fit logistic regression with gradient descent and get the same answer as scikit-learn.

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as NumPy (`np.`), pandas (`pd.`), and scikit-learn, do everyday tasks you'll do yourself: loading data, fitting a model, writing a gradient descent loop. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_4_demos_helpers` draw a plot or build an interactive demo. That file was written for this course, and its plotting code isn't the point. You never need to open it. Ignore this kind of code and focus on how the resulting plots, demos, etc fit into the lesson.

## Setup

**Code worth reading.** Run this cell first. It imports the libraries we use, with their usual short names, and loads the two data sets: ISL's `Default` (for logistic regression) and `Advertising` (for linear regression).

In [ ]:
import numpy as np
import pandas as pd
from ISLP import load_data

Default = load_data('Default')                     # ISLP ships this data set
ad = pd.read_csv('Advertising.csv', index_col=0)   # index_col=0: the first column is just row numbers

## 1 · Finding the bottom of a curve

Gradient descent is a method for finding the minima of a function. Before we use it to fit models, we'll develop the idea for a simple function with only one minimum:

$$
f(x) = (x - 4)^2 + 2
$$

Looking at the graph, you can see the lowest point is at $x = 4$. But imagine you can't see the whole curve. All you know is where you're standing, the point $(x, f(x))$, and how steep the curve is right there: the slope of the **tangent line**, the straight line that just touches the curve at that point.

### Demo 1: Which way is downhill?

The point starts at $x = 6.3$, to the right of the minimum. Move the slider and watch the tangent line and its slope.

In [ ]:
from ch_4_demos_helpers import tangent_slope_demo

tangent_slope_demo(size_percent=85)

**Ask:** When $x$ is to the right of the minimum, what sign does the slope have? When $x$ is to the left of the minimum?

<details>
<summary>Answer</summary>

To the right of the minimum, the slope is **positive**: the curve rises as you move right. To the left, the slope is **negative**: the curve falls as you move right. Exactly at the minimum, $x = 4$, the tangent line is flat and the slope is 0.

</details>

**The goal.** Move the point toward the bottom by changing its $x$ coordinate, using *only* the slope of the tangent line where the point is now. You don't get to see the rest of the curve.

**Ask:** Given the x-coordinate and the slope (of the tangent line), how should you change x to find the minimum?

<details>
<summary>Answer</summary>

**Move opposite to the sign of the slope.**

- If the slope is positive, the curve rises to the right, so downhill is to the left: move $x$ in the negative $x$ direction (left).
- If the slope is negative, the curve falls to the right, so downhill is to the right: move $x$ in the positive $x$ direction (right).
- If the slope is 0, the ground is flat: you're at the bottom, so stop.

</details>

## 2 · A first try: fixed-size steps

Here's a first algorithm built on that rule. Choose a step size, say 0.25, and repeat:

1. Start at some $x$, and find the point on the curve, $(x, f(x))$.
2. Find the slope of the tangent line there, and note its sign.
3. Move $x$ by the step size in the downhill direction: left if the slope is positive, right if it's negative.
4. Find the new point on the curve, and go back to step 2.

To find the slopes, we use the derivative. For $f(x) = (x - 4)^2 + 2$, the slope of the tangent line at $x$ is

$$
f'(x) = 2(x - 4)
$$

Notice that this algorithm uses only the slope's **sign**, to pick a direction. It ignores the slope's size.

<a id="demo-2"></a>

### Demo 2: Stepping downhill by a fixed amount

Click **Step** to take one step. The box above the plot shows the arithmetic for the latest step, and the table below the plot records every step. **10 steps** takes ten at once, and **Random start** picks a new starting $x$.

In [ ]:
from ch_4_demos_helpers import fixed_step_demo

fixed_step_demo(size_percent=85)

**Try this.**

- **Step size 0.25,** starting from $x = 6.3$. Keep clicking after step 9. The point reaches 4.05, then bounces between 4.05 and 3.8 forever. It can never land on 4, because every step is exactly 0.25 long.
- **Step size 1.5.** The first step gets close, then the point overshoots back and forth between 4.8 and 3.3, which is much worse.
- **Step size 0.05.** It takes 46 steps to reach the bottom from $x = 6.3$. (It lands exactly on 4 only because $6.3 - 4 = 2.3$ is a whole number of 0.05 steps. Click **Random start**: from most starting points, it bounces at the bottom too, just in tinier steps.) From farther away, it could take hundreds of steps.

**The problem.** Every step is the same size, no matter where we are. Far from the bottom, we want big steps so we get there quickly. Near the bottom, we want small steps so we don't overshoot. One fixed size can't do both. And we have no good way to choose the size in advance: a step that's tiny for one curve is huge for a wider or narrower one.

**Ask:** Far from the minimum we'd like big steps, and near it small ones. Look at the slope column in the table. Is there something we already compute at every step that's big when we're far from the minimum and small when we're close?

<details>
<summary>Answer</summary>

**The size of the slope.** Far up the side of the curve, it's steep, so the slope is large. Near the bottom, the curve flattens out and the slope gets small; at the bottom, it's 0. For this curve, the slope $2(x - 4)$ is exactly twice the distance from the minimum.

That suggests letting the slope set the size of the step, not just its direction. Steep ground would mean a big step, and flat ground a small one, so the steps would shrink on their own as we approach the bottom.

</details>

## 3 · Scaling the step by the slope <a id="section-3"></a>

Section 2 ended with an idea: let the slope set the *size* of each step, not just its direction. So instead of moving a fixed distance, we move by a multiple of the slope:

$$
\Delta x = -\eta \, f'(x), \qquad x_{\text{new}} = x + \Delta x = x - \eta \, f'(x)
$$

Here $\Delta x$ ("delta x") is the change in $x$ for one step. Each part of the rule has a job:

- **The slope $f'(x)$ sets the size.** Steep ground means a big step and flat ground a small one. At the bottom, the slope is 0, so the steps shrink away on their own.
- **The minus sign sets the direction.** A positive slope makes $\Delta x$ negative, a step to the left, which is downhill. That's the rule from Section 1, built into the formula: no more "if the slope is positive, move left."
- **$\eta$** (the Greek letter eta) is a small positive number we choose. It's no longer a step size, because the step now changes every time. Instead, it sets how big a step we take for a given slope. It's called the **learning rate**.

This rule is **gradient descent**. (The *gradient* is what the slope is called when there's more than one variable, which comes later in this notebook.)

<a id="demo-3"></a>

### Demo 3: Steps proportional to the slope

**Try it on paper first.** With $\eta = 0.25$, starting at $x = 7.4$, fill in three rows:

| $x$ | slope $2(x - 4)$ | $\Delta x = -\eta \times \text{slope}$ | new $x$ |
| --- | --- | --- | --- |
| 7.4 | | | |
| | | | |
| | | | |

Then click **Step** to check each row. To compare with Section 2, click **fixed step**: it restarts from the same $x$ with fixed steps of 0.25.

In [ ]:
from ch_4_demos_helpers import scaled_step_demo

scaled_step_demo(size_percent=85)

**Try this.**

- **$\eta = 0.25$.** The steps are $-1.7$, then $-0.85$, then $-0.425$: each one is half the one before. Nobody told them to shrink. They shrink because the slope shrinks as we approach the bottom. After 13 steps, $x$ is within 0.0005 of 4. Fixed steps of 0.25 from the same start are bouncing between 3.9 and 4.15 by then.
- **$\eta = 0.5$.** It lands exactly on the bottom in one step. (That's a coincidence of this particular curve.)
- **$\eta = 0.9$.** It overshoots to the other side, but each overshoot is smaller than the last, so it zigzags in.
- **$\eta = 1$.** It bounces between 7.4 and 0.6 forever.
- **$\eta = 1.05$.** Each overshoot is bigger than the last, and it runs off the plot.

**We still have to choose $\eta$.** Scaling by the slope fixes the problem from Section 2: near the bottom, the steps get small without our doing anything. But too small an $\eta$ is still slow, and too big an $\eta$ still overshoots or blows up. Section 7 comes back to choosing $\eta$, and to deciding when to stop.

**Ask:** With $\eta = 0.25$, every step covers exactly half the remaining distance to the bottom. Why half? (Hint: compare the slope $2(x - 4)$ with the distance from $x$ to 4.)

<details>
<summary>Answer</summary>

The slope $2(x - 4)$ is twice the distance from $x$ to 4. So the step is

$$
\Delta x = -0.25 \times 2(x - 4) = -0.5\,(x - 4),
$$

which is half the distance to 4, in the direction of 4. After each step, half the distance is left.

The same reasoning works for any $\eta$: each step multiplies the distance to 4 by $(1 - 2\eta)$. With $\eta = 0.5$, that's 0, so one step lands on the bottom. With $\eta = 0.9$, it's $-0.8$: we land on the other side, a bit closer. With $\eta = 1$, it's $-1$: the other side, just as far away, forever. With $\eta = 1.05$, it's $-1.1$: the other side, farther away every time.

</details>

## 4 · Three ways to fit a model

<a id="demo-4"></a>

### Demo 4: Formula, counting, or searching

Fitting always means choosing the coefficients that make the model predict the data best. What changes from model to model is *how* we find them.

| Linear regression | Naive Bayes | Logistic regression |
| --- | --- | --- |
| **A formula.** Least squares gives the best coefficients directly from the data, in one calculation. | **Counting.** The priors and the class-by-class estimates come straight from counts and averages in each class. | **Searching.** There's no formula for the best coefficients, so the computer tries values and improves them until it can't do better. |

**Code worth reading.** Here's the logistic regression from Chapter 4's Notebook 2 (*Fitting and Judging a Classifier*), predicting default from standardized balance. We standardize by hand, fit with scikit-learn, and read off the coefficients.

`C=np.inf` turns off a penalty that scikit-learn adds by default (Chapter 6 covers penalties), so we get the plain maximum likelihood fit that ISL reports.

In [ ]:
from sklearn.linear_model import LogisticRegression

balance = Default['balance']
x = ((balance - balance.mean()) / balance.std()).to_numpy()   # standardized balance
y = (Default['default'] == 'Yes').astype(int).to_numpy()      # 1 = defaulted, 0 = didn't

model = LogisticRegression(C=np.inf)
model.fit(x.reshape(-1, 1), y)    # scikit-learn wants a 2-D table of features: here, one column

print(f'Intercept  β₀ = {model.intercept_[0]:.4f}')
print(f'Slope      β₁ = {model.coef_[0, 0]:.4f}')
print()
print('Where did these numbers come from? Today we find out.')

**Fitting as minimizing a loss.** To search, the computer needs a score to improve. We call it the **loss**, written $L(\beta)$: a number that measures how badly the model with coefficients $\beta$ fits the data. Lower is better. (You'll also see it called the *cost* or the *objective*.)

For linear regression, the loss is the squared error you met in Chapter 3: least squares picks the line with the smallest total squared error. Logistic regression uses a different loss, built from the likelihood in Notebook 2. We'll come back to it in Section 10, once gradient descent is fully built. Until then, every example uses squared error.

**Ask:** In Chapter 3, least squares chose the line with the smallest RSS. In the language of this section, what is the loss function, and what's our goal?

<details>
<summary>Answer</summary>

The loss is the RSS: the total squared error of the line's predictions. Our goal is to find the coefficients $\beta_0$ and $\beta_1$. Every pair of $\beta$-values gives a different line with its own loss, and **fitting** means finding the pair producing the lowest loss.

</details>

## 5 · Two spaces: the data and the parameters

We often see plots with **data** plotted on the axes. A *loss plot is different*: it puts a **parameter** (such as slope, intercept, etc) on the horizontal axis and the loss on the vertical axis. Each point on a loss plot comes from a different model (specified by it's unique parameter values).

### Demo 5: Two spaces, linked

We use the `Advertising` data: sales (in thousands of units) against the TV advertising budget (in thousands of dollars). The model is $\text{sales} \approx \beta_0 + \beta_1 \cdot \text{TV}$, and the loss is the mean squared error:

$$
L(\beta_0, \beta_1) = \frac{1}{n}\sum_{i=1}^{n} \big(y_i - \beta_0 - \beta_1 x_i\big)^2
$$

<details>
<summary>Reading the mean squared error formula</summary>

**The simpler form.** For each observation $i$, the model makes a prediction $\hat{y}_i$ ("y-hat"). The error, or residual, is the actual value minus the prediction, $y_i - \hat{y}_i$. The mean squared error squares each error and averages them:

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} \big(y_i - \hat{y}_i\big)^2
$$

**Substituting our model.** Our model predicts $\hat{y}_i = \beta_0 + \beta_1 x_i$. Put that in place of $\hat{y}_i$:

$$
\frac{1}{n}\sum_{i=1}^{n} \big(y_i - (\beta_0 + \beta_1 x_i)\big)^2 = \frac{1}{n}\sum_{i=1}^{n} \big(y_i - \beta_0 - \beta_1 x_i\big)^2
$$

We write the result as $L(\beta_0, \beta_1)$ because the data are fixed: the only things we can change are the coefficients, so the loss is a function of them.

**What the symbols mean.**

- $n$: the number of observations (here, 200 markets), and $i = 1, \dots, n$ numbers them.
- $x_i$ and $y_i$: market $i$'s TV budget and its actual sales.
- $\hat{y}_i = \beta_0 + \beta_1 x_i$: the sales the line predicts for market $i$.
- $y_i - \hat{y}_i$: how far the prediction misses, positive if the line is too low and negative if it's too high.
- The square makes every miss count as positive, and makes big misses count much more than small ones.
- $\frac{1}{n}\sum_{i=1}^{n}$: add up over all the markets and divide by $n$, which is an average.

**In words:** $L$ is the average squared miss of the line. The smaller it is, the closer the line passes to the points.

</details>

This is Chapter 3's RSS divided by the number of observations $n$. Dividing by $n$ doesn't move the lowest point, so it picks the same best line, and it keeps the numbers a readable size.

To keep things to one parameter, we fix the intercept $\beta_0$ at its least-squares value and let only the slope $\beta_1$ move.

**Code worth reading.** Linear regression is the "formula" case from Section 4: scikit-learn finds the least-squares coefficients in one call.

In [ ]:
from sklearn.linear_model import LinearRegression

least_squares = LinearRegression()
least_squares.fit(ad[['TV']], ad['sales'])   # double brackets keep TV as a one-column table

print(f'Intercept  β₀ = {least_squares.intercept_:.4f}')
print(f'Slope      β₁ = {least_squares.coef_[0]:.4f}')

The demo below holds $\beta_0$ at that value. Move the slider and watch both plots.

In [ ]:
from ch_4_demos_helpers import two_spaces_demo

two_spaces_demo(ad)

**Ask:** Where on the right curve is the best line? What does the left plot look like there?

<details>
<summary>Answer</summary>

The best line is at the **bottom** of the curve, where the loss is smallest, at about $\beta_1 = 0.048$. That's the least-squares slope. On the left, the line runs through the middle of the cloud of points. Move away from the bottom in either direction, and the line on the left tilts too steeply or too flatly.

</details>

## 6 · Walking downhill by hand <a id="section-6"></a>

Suppose we can't see the whole loss curve, only the ground under our feet. The **slope** of the loss tells us two things:

- **Its sign says which way is downhill.** A negative slope means the loss falls as $\beta$ grows, so downhill is to the right. A positive slope means downhill is to the left.
- **Its size says how steep it is.** A big slope means we're far up the side; a slope near 0 means we're near the bottom.

The slope of the loss is its derivative, $\dfrac{dL}{d\beta}$.

To practice, we use a loss whose bottom we already know, so we can check our work:

$$
L(\beta) = (\beta - 3)^2, \qquad \frac{dL}{d\beta} = 2(\beta - 3)
$$

The minimum is at $\beta = 3$. Pretend we don't know that, start at $\beta = 0$, and walk downhill.

**The update rule.** Each step moves $\beta$ against the slope, by an amount proportional to the slope:

$$
\beta^{(t+1)} = \beta^{(t)} - \eta \, \frac{dL}{d\beta}\Big|_{\beta^{(t)}} \qquad \text{(one parameter)}
$$

In words: **new value = old value $-$ (learning rate) $\times$ (slope at the old value).** Here $t$ counts the steps, and $\beta^{(t)}$ is the value after step $t$.

- **The minus sign** sends us downhill. If the slope is negative, subtracting it makes $\beta$ bigger, which moves us to the right, the downhill direction.
- **The learning rate $\eta$** (the Greek letter eta) is a small positive number we choose. It sets how far each step goes for a given slope.

> **A note on symbols.** Different books and libraries use different letters for the same things. ISL's own gradient descent section (§10.7) writes the parameters as $\theta$, the learning rate as $\rho$, and the loss as $R(\theta)$. Many machine learning courses use $\theta$ and $\alpha$. scikit-learn calls the learning rate `eta0` and uses `alpha` for something unrelated (the strength of regularization). We use $\beta$ for the parameters, to match the regression chapters, and $\eta$ for the learning rate, to match scikit-learn. The letters don't matter. What matters is the relationship: **new value = old value $-$ (learning rate) $\times$ (slope).**

### Demo 6: Walking downhill by hand

**Try it on paper first.** With $\eta = 0.25$ and $\beta^{(0)} = 0$, fill in three rows:

| $t$ | $\beta^{(t)}$ | slope $2(\beta^{(t)} - 3)$ | $\eta \times \text{slope}$ | $\beta^{(t+1)}$ |
| --- | --- | --- | --- | --- |
| 0 | 0 | | | |
| 1 | | | | |
| 2 | | | | |

Then click **Step** to check each row. The black segment is the tangent line (its tilt is the slope), and the arrow points to where the next step lands.

In [ ]:
from ch_4_demos_helpers import descent_by_hand_demo

descent_by_hand_demo()

Look down the "$\eta \times \text{slope}$" column: each step is half the size of the one before. Nobody told the steps to shrink. They shrink on their own because the slope flattens as we approach the bottom.

**Ask:** The slope is negative. Why does $\beta$ go *up*?

<details>
<summary>Answer</summary>

Because of the minus sign in the update. A negative slope means the loss falls to the right, so downhill is toward bigger $\beta$. Subtracting a negative number adds: $0 - 0.25 \times (-6) = 1.5$.

</details>

## 7 · The learning rate, and when to stop

The learning rate $\eta$ is ours to choose, and the choice matters. For our practice loss, each step multiplies the distance to 3 by $(1 - 2\eta)$. So a small $\eta$ closes the gap slowly, $\eta = 0.5$ closes it in one step, and a large $\eta$ overshoots to the other side.

**When to stop.** In real problems we don't know where the bottom is, so we can't stop "at 3." A common rule is to **stop when the loss barely changes from one step to the next**, say by less than 0.001, and to also set a maximum number of steps in case that never happens.

<a id="demo-7"></a>

### Demo 7: The learning rate, and when to stop

Each preset has a known behavior. Before clicking, predict what the path and the loss will do.

In [ ]:
from ch_4_demos_helpers import learning_rate_demo

learning_rate_demo()

**Ask:** If you saw only the right-hand plot, how would you tell 'too small' from 'too big'?

<details>
<summary>Answer</summary>

With $\eta$ too small, the loss falls steadily but slowly: a gentle, smooth slope downward. With $\eta$ too big, the loss either jumps up and down on its way down (zigzagging in) or climbs and keeps climbing (blowing up). A loss that ever goes **up** is a sign that $\eta$ is too big.

</details>

## 8 · Two parameters: the loss surface and its contour map

Now let both $\beta_0$ and $\beta_1$ move. The loss depends on two numbers, so instead of a curve it's a **surface**: a bowl over the $(\beta_0, \beta_1)$ floor. We keep using `Advertising`, but with TV **standardized** (converted to z-scores), which makes the bowl nicely round. Section 9 shows what happens without that.

<a id="demo-8a"></a>

### Demo 8a: From a surface to a contour map

A 3-D surface is hard to read on a flat screen, so from here on we draw **contour plots** instead. A contour plot is the surface seen from directly above. Slice the bowl at one height, and the edge of the slice is one contour line: every $(\beta_0, \beta_1)$ on it gives the same loss. Rings close together mean the ground is steep; rings far apart mean it's nearly flat. The center of the rings is the bottom of the bowl.

You saw a pair like this once before, in the Chapter 3 regression demos ("Finding the global minimum"). Drag the surface to rotate it, and use the slider to slice it at one height.

In [ ]:
from ch_4_demos_helpers import surface_and_contour

surface_and_contour(ad)

**Ask:** Where is the surface steepest? How can you tell from the contour plot alone?

<details>
<summary>Answer</summary>

The surface is steepest far from the bottom, out toward the edges. On the contour plot, that's where the contour lines are **closest together**: the loss climbs a full level in a short distance. Near the center, the lines are far apart, because the bowl is nearly flat there.

</details>

<a id="demo-8b"></a>

### Demo 8b: Partial derivatives as slices

With two parameters, there are two slopes. Freeze $\beta_1$ and move only $\beta_0$: the loss traces out an ordinary one-parameter curve, and its slope is called the **partial derivative** $\partial L / \partial \beta_0$. Freezing $\beta_0$ and moving only $\beta_1$ gives the other one, $\partial L / \partial \beta_1$.

The **gradient** is the list of those slopes, stacked into a vector:

$$
\nabla L(\beta) = \left( \frac{\partial L}{\partial \beta_0}, \; \frac{\partial L}{\partial \beta_1} \right)
$$

Drawn as an arrow on the contour plot, the gradient points in the direction where the loss climbs fastest. So $-\nabla L$, the opposite direction, is the steepest way downhill.

For the mean squared error, the two slopes are

$$
\frac{\partial L}{\partial \beta_0} = -\frac{2}{n}\sum_i \big(y_i - \beta_0 - \beta_1 x_i\big), \qquad \frac{\partial L}{\partial \beta_1} = -\frac{2}{n}\sum_i \big(y_i - \beta_0 - \beta_1 x_i\big)\,x_i
$$

Read each one as an average of **(prediction error) $\times$ (feature)**, where the "feature" for the intercept $\beta_0$ is just 1. Move the sliders and watch the two slices and the arrows change. In the plot, the arrows are drawn shorter than their true length so they fit.

In [ ]:
from ch_4_demos_helpers import partial_slopes_demo

partial_slopes_demo(ad)

**Ask:** Each small plot is a one-parameter problem you already know how to solve. What does the arrow add?

<details>
<summary>Answer</summary>

It combines the two slopes into a single direction. Each small plot only says which way to move one parameter with the other held still. The arrow $-\nabla L$ says how to move **both at once**: how much of the step goes to $\beta_0$ and how much to $\beta_1$. That combined direction is the steepest way downhill.

</details>

## 9 · Descent on a contour plot, and why scaling matters <a id="section-9"></a>

The update rule for two (or any number of) parameters is the same rule as before, applied to every parameter at once, each with its own partial derivative:

$$
\beta_j^{(t+1)} = \beta_j^{(t)} - \eta \, \frac{\partial L}{\partial \beta_j}\Big|_{\beta^{(t)}} \text{ for every } j, \qquad\text{or}\qquad \beta^{(t+1)} = \beta^{(t)} - \eta \, \nabla L(\beta^{(t)})
$$

Every step goes in the direction of $-\nabla L$, so it leaves its starting point straight across the contour lines, at a right angle to them.

<a id="demo-9a"></a>

### Demo 9a: The two slopes when a feature needs rescaling

Before running gradient descent, look at the slopes it will use. This is the same regression as Demo 8b, with one change: TV is in hundreds of thousands of dollars (TV divided by 100), and it isn't standardized. Its values run from about 0 to 3, all positive, with an average of about 1.5. The data and the best line are the same; only the feature's units changed.

The bowl is now a long, tilted valley. Move the sliders and watch three things: the two slopes, the ratio between them (in the readout), and where the downhill arrow points. Try these points:

- $(\beta_0, \beta_1) = (0, 0)$, where gradient descent will start in Demo 9b.
- A few other points on the walls of the valley.
- Points on the valley floor, far from the star, such as $\beta_0 = 10.5$, $\beta_1 = 3$.

In [ ]:
from ch_4_demos_helpers import unscaled_slopes_demo

unscaled_slopes_demo(ad)

**Ask:** What do you notice about the two slopes and the downhill arrow? Can you guess what problems this will cause when gradient descent starts at $(0, 0)$?

<details>
<summary>Answer</summary>

- **On the walls, the downhill arrow doesn't point at the star.** Both slopes have the same sign, and $\partial L / \partial \beta_1$ is roughly 1.5 to 3 times $\partial L / \partial \beta_0$. So the arrow points across the valley, toward the floor, rather than along the valley toward the minimum. *Prediction:* the first steps rush across the valley, overshoot onto the opposite wall, and come back, zigzagging.
- **On the valley floor, both slopes are small, even far from the star.** At $\beta_0 = 10.5$, $\beta_1 = 3$, they're about 1.8 and 0.04. Gradient descent takes steps proportional to the slopes. *Prediction:* once the path reaches the floor, it crawls slowly toward the minimum.
- **The two slices curve very differently.** The "move $\beta_1$ only" curve is much narrower than the "move $\beta_0$ only" curve. One learning rate $\eta$ has to work for both. *Prediction:* an $\eta$ small enough not to overshoot in the steep direction is too small to make quick progress in the gentle one.

Compare Demo 8b, where TV was standardized: there, the downhill arrow points straight at the star from every point.

</details>

<a id="demo-9b"></a>

### Demo 9b: Descent on a contour plot, and why scaling matters

Now check your predictions. Compare two versions of the same regression:

- **Standardized TV:** TV as z-scores, as in Section 8. The contours are nearly circles.
- **Unscaled TV:** TV in hundreds of thousands of dollars (TV divided by 100), not centered. The bowl stretches into a long, tilted valley.

Each version has a **largest stable learning rate**: above it, the steps overshoot more each time and the descent blows up, as $\eta = 1.05$ did in Demo 7. The presets are 10%, 50%, and 95% of that limit. Both axes use the same scale, so right angles look like right angles. Use the step slider to replay a path.

In [ ]:
from ch_4_demos_helpers import scaling_demo

scaling_demo(ad)

At 50% of the limit, standardized TV reaches the bottom in **1 step**, and unscaled TV takes **52**. Same data, same model, same best fit; only the units of the feature changed. This is why we scale features before running gradient descent. (At 95%, both versions overshoot back and forth, so that preset isn't a fair comparison of the two.)

**Ask:** Why does each step cross the contour lines at a right angle? Why does that make it zigzag in a narrow valley?

<details>
<summary>Answer</summary>

Along a contour line the loss doesn't change at all, so the steepest way down points straight across the lines, at a right angle. In a round bowl, "straight across the lines" points at the center. In a long, narrow valley, it points mostly across the valley, toward the nearest wall, not along the valley toward the bottom. So each step crosses the valley, overshoots, and the next step crosses back: a zigzag that makes slow progress along the valley floor.

</details>

## 10 · Payoff: fitting logistic regression ourselves

Back to the question from Section 4: where did scikit-learn's coefficients come from? We now have everything we need to find them ourselves, except logistic regression's loss.

That loss is the **average negative log-likelihood**: Notebook 2's log-likelihood with its sign flipped, making a lower score "better" than a high score (the log-likelihood surface is a hill, so we "flip" it over with the negative sign, making it like a bowl, where the lowest point corresponds to the best model parameters), and averaged over the $n$ customers. Flipping the sign turns the likelihood's highest point into the loss's lowest point, so walking downhill on this loss is the same as walking uphill on the likelihood. Written out, the loss is

$$
L(\beta_0, \beta_1) = -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
$$

Here $p_i$ is the model's predicted probability that customer $i$ defaults, and $y_i$ is 1 if they defaulted and 0 if not. Only one of the two terms in the brackets is ever switched on: for a defaulter it's $\ln p_i$, and for everyone else it's $\ln(1 - p_i)$. Either way, it's the log of the probability the model gave to what actually happened. scikit-learn computes $L$ for us with `log_loss`.

<details>
<summary>Where the negative log-likelihood comes from</summary>

**Start from Notebook 2.** The likelihood multiplies, over every customer, the probability the model gave to what actually happened (ISL equation 4.5). Write $p_i$ for the model's predicted probability that customer $i$ defaults, and $y_i$ for 1 if they defaulted and 0 if not:

$$
\ell(\beta_0, \beta_1) = \prod_{i:\,y_i = 1} p_i \prod_{i:\,y_i = 0} (1 - p_i)
$$

The first product runs over the customers who defaulted, and the second over those who didn't.

**Step 1: one product instead of two.** Because $y_i$ is either 1 or 0, the factor $p_i^{\,y_i}(1 - p_i)^{1 - y_i}$ equals $p_i$ for a defaulter and $1 - p_i$ for everyone else. (Anything raised to the power 0 is 1, so one of the two pieces always switches off.) That lets us write both products as one, over all customers:

$$
\ell(\beta_0, \beta_1) = \prod_{i=1}^{n} p_i^{\,y_i}(1 - p_i)^{1 - y_i}
$$

**Step 2: take the log.** The log of a product is the sum of the logs, and $\ln(a^b) = b \ln a$:

$$
\ln \ell(\beta_0, \beta_1) = \sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
$$

*Why take the log?* There are two reasons, and both come from turning the product into a sum.

**Reason 1: slopes.** Gradient descent needs the slope of the loss with respect to each $\beta_j$. To keep the notation short, write $f_i = p_i^{\,y_i}(1 - p_i)^{1 - y_i}$ for the probability the model gave to what actually happened to customer $i$, and write $f_i'$ for its slope with respect to $\beta_j$. Every $f_i$ depends on $\beta_j$, because every $p_i$ does.

With just three customers, the product rule gives the slope of the likelihood as

$$
\frac{\partial}{\partial \beta_j}\big(f_1 f_2 f_3\big) = f_1' f_2 f_3 + f_1 f_2' f_3 + f_1 f_2 f_3'
$$

Each term has one customer's slope multiplied by *every other customer's* probability. With 10,000 customers, that's 10,000 terms, each a product of 10,000 factors. No customer's contribution can be worked out without all the others.

The slope of the log-likelihood is different, because the slope of a sum is the sum of the slopes:

$$
\frac{\partial}{\partial \beta_j}\big(\ln f_1 + \ln f_2 + \ln f_3\big) = (\ln f_1)' + (\ln f_2)' + (\ln f_3)'
$$

Here each term involves only one customer. Adding doesn't tangle the customers together the way multiplying does. Each term also simplifies a lot: $(\ln f_i)'$ works out to $(y_i - p_i)\,x_{ij}$, customer $i$'s prediction error times their feature. After Step 3 flips the sign and averages, that becomes the slope formula below the derivation, with $(p_i - y_i)$ in place of $(y_i - p_i)$.

**Reason 2: arithmetic.** Multiplying 10,000 probabilities gives a number far too small for a computer to store. At scikit-learn's coefficients the likelihood is about $e^{-798} \approx 10^{-347}$. That's smaller than the tiniest number Python's floats can hold, so the product rounds to exactly 0. Adding 10,000 logs gives an ordinary number instead.

Taking the log doesn't change which coefficients are best: a bigger likelihood always has a bigger log, so the highest point is in the same place.

This is the log-likelihood from Notebook 2: for each customer, the log of the probability given to what happened, all added up. At scikit-learn's coefficients it's about $-798$.

**Step 3: turn it into a loss.** Flip the sign, so that lower is better, and divide by $n$, so that the size doesn't depend on how many customers there are:

$$
L(\beta_0, \beta_1) = -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
$$

Neither change moves the best coefficients. At scikit-learn's coefficients, $L$ is about $798 / 10{,}000 \approx 0.08$. This average is what `log_loss` computes.

</details>

Its slopes are

$$
\frac{\partial L}{\partial \beta_j} = \frac{1}{n}\sum_i \big(p_i - y_i\big)\,x_{ij}
$$

where $x_{i0} = 1$ for the intercept and $x_{i1}$ is customer $i$'s standardized balance. This has the same shape as the linear regression slopes in Section 8: an average of **(prediction error) $\times$ (feature)**. Here the prediction is the probability $p_i$.

### Demo 10: Gradient descent on the `Default` data

We start at $\beta_0 = \beta_1 = 0$ and take 2,000 steps with $\eta = 4$. That's a much bigger learning rate than Demo 7 could handle, but this loss's bowl is much gentler, so the steps stay stable.

**Code worth reading.** This is the whole algorithm. We put a column of 1s next to balance, so that `X @ beta` computes $\beta_0 + \beta_1 x_i$ for every customer at once. Then each pass through the loop computes the predicted probabilities, records the loss with `log_loss`, computes the list of slopes with the formula above, and applies the update rule. It reuses `x` and `y` from Demo 4.

In [ ]:
from sklearn.metrics import log_loss

X = np.column_stack([np.ones(len(x)), x])   # column 0 is all 1s (the intercept); column 1 is balance
n = len(y)
eta = 4.0

beta = np.zeros(2)          # start at (β₀, β₁) = (0, 0)
beta_path = [beta]          # every (β₀, β₁) we visit, for plotting
losses = []                 # the loss at each step, for plotting

for t in range(2000):
    p = 1 / (1 + np.exp(-(X @ beta)))                          # predicted probabilities p_i
    losses.append(log_loss(y, p))                              # average negative log-likelihood
    grad = X.T @ (p - y) / n                                   # (∂L/∂β₀, ∂L/∂β₁)
    beta = beta - eta * grad                                   # the update rule
    beta_path.append(beta)

<a id="code-explanations-10"></a>

**Code Explanations:**

**What the loop computes.** Each pass through the loop applies the update rule from Section 9, using the logistic regression slopes from above:

$$
\underbrace{\begin{bmatrix} \beta_0 \\ \beta_1 \end{bmatrix}}_{\texttt{beta}}
\leftarrow
\begin{bmatrix} \beta_0 \\ \beta_1 \end{bmatrix}
- \eta \;
\underbrace{\frac{1}{n} \begin{bmatrix} \sum_i (p_i - y_i) \cdot 1 \\ \sum_i (p_i - y_i)\, x_i \end{bmatrix}}_{\texttt{grad}}
\qquad \text{where} \qquad
\underbrace{p_i}_{\texttt{p}} = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_i)}}
$$

The code builds this in four pieces, one line each:

1. $\beta_0 + \beta_1 x_i$ for every customer: `X @ beta`.
2. $p_i$ for every customer: `p = 1 / (1 + np.exp(-(X @ beta)))`.
3. The two sums, divided by $n$: `grad = X.T @ (p - y) / n`.
4. The update itself: `beta = beta - eta * grad`.

Pieces 1 and 3 use matrix multiplication, so the rest of this section breaks those two down.

Here $n = 10{,}000$ customers, and $x_i$ is customer $i$'s standardized balance. The explanations below track each array's **shape**, which NumPy reports with `.shape`: `(10000, 2)` means 10,000 rows and 2 columns, and `(2,)` means a plain list of 2 numbers.

**Why we need a column of 1s.** For each customer, the model needs $\beta_0 + \beta_1 x_i$. Matrix multiplication can compute this for every customer at once, but it only knows one move: multiply each column by its coefficient and add up the results. $\beta_1$ has a column to multiply, the balances. $\beta_0$ doesn't, so we give it one. Writing $\beta_0$ as $\beta_0 \times 1$ shows that the intercept is the coefficient on a "feature" that equals 1 for every customer. That column of 1s is the $x_{i0} = 1$ in the slope formula above. Without it, `X @ beta` would compute only $\beta_1 x_i$, and the model would have no intercept.

**What `X @ beta` does.** `@` is matrix multiplication. `X` has shape `(10000, 2)`: one row per customer and one column per coefficient. `beta` has shape `(2,)`: one number per coefficient. This is piece 1: it computes the $\beta_0 + \beta_1 x_i$ inside $p_i$, for all customers at once. Each row of `X` is multiplied entry by entry with `beta`, and the products are added:

$$
X\beta =
\begin{bmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_n \end{bmatrix}
\begin{bmatrix} \beta_0 \\ \beta_1 \end{bmatrix}
=
\begin{bmatrix} 1 \cdot \beta_0 + x_1 \beta_1 \\ 1 \cdot \beta_0 + x_2 \beta_1 \\ \vdots \\ 1 \cdot \beta_0 + x_n \beta_1 \end{bmatrix}
$$

The shapes follow a rule: `(10000, 2) @ (2,)` gives `(10000,)`. The two inner sizes must match (2 and 2), because each customer's 2 features pair up with the 2 coefficients. Those inner sizes disappear in the sum, leaving one number per customer. `1 / (1 + np.exp(-(X @ beta)))` then applies the logistic function to each of those 10,000 numbers separately, so `p` also has shape `(10000,)`: one predicted probability per customer.

**The gradient line, `grad = X.T @ (p - y) / n`.** This is piece 3: it computes the two sums inside `grad` in the formula at the top. `p - y` has shape `(10000,)`: each customer's prediction error, $p_i - y_i$. The slope formula needs two sums over customers: the errors times 1 for $\beta_0$, and the errors times $x_i$ for $\beta_1$. `X.T` is `X` **transposed**, meaning its rows and columns swapped, so its shape is `(2, 10000)`. Row 0 is all 1s and row 1 is all the balances:

$$
X^{T}(p - y) =
\begin{bmatrix} 1 & 1 & \cdots & 1 \\ x_1 & x_2 & \cdots & x_n \end{bmatrix}
\begin{bmatrix} p_1 - y_1 \\ p_2 - y_2 \\ \vdots \\ p_n - y_n \end{bmatrix}
=
\begin{bmatrix} \sum_i (p_i - y_i) \cdot 1 \\ \sum_i (p_i - y_i)\, x_i \end{bmatrix}
$$

These are the two sums in `grad` at the top. Dividing by $n$ gives `grad`: exactly $\partial L / \partial \beta_0$ and $\partial L / \partial \beta_1$ from the slope formula.

*Why the transpose?* Matrix multiplication adds up along the inner size, and we want to add up over customers. `X @ (p - y)` would be `(10000, 2) @ (10000,)`. The inner sizes, 2 and 10,000, don't match, so NumPy raises an error. Transposing puts the customers in the inner position: `(2, 10000) @ (10000,)` gives `(2,)`. The 10,000 customers are summed away, leaving one slope per coefficient. That's the same shape as `beta`, which the update `beta - eta * grad` needs.

The two multiplications go in opposite directions. `X @ beta` goes from 2 coefficients to 10,000 predictions, and `X.T @ (p - y)` goes from 10,000 errors back to 2 slopes.


Here's the loss at each step:

In [ ]:
from ch_4_demos_helpers import plot_logistic_loss

plot_logistic_loss(losses, eta);

The loss drops quickly at first, then flattens out as the slopes shrink near the bottom.

**Code worth reading.** Here's where gradient descent ended, next to scikit-learn's answer from Demo 4, in a small pandas table:

In [ ]:
sklearn_beta = np.array([model.intercept_[0], model.coef_[0, 0]])

pd.DataFrame([beta, sklearn_beta, beta - sklearn_beta],
             index=['gradient descent', 'scikit-learn', 'difference'],
             columns=['β₀', 'β₁']).round(4)

The two agree to about three decimal places. Here's the path on a contour plot of this loss. It's a single bowl too, just a stretched one.

In [ ]:
from ch_4_demos_helpers import plot_logistic_descent_path

plot_logistic_descent_path(beta_path, x, y, best=sklearn_beta);

The numbers from Demo 4 came from a search like this one. scikit-learn uses a faster search than plain gradient descent, but it ends at the same bottom of the same bowl.

**Ask:** How do we know gradient descent found *the* best coefficients, and not just some low spot?

<details>
<summary>Answer</summary>

The losses for linear and logistic regression are single bowls with exactly one bottom, as every contour plot in this notebook shows. Walking downhill in a single bowl can only end at that one bottom. (Neural network losses aren't single bowls; that's Week 9.)

</details>

## 11 · Summary, and what's next

- Some models are fit by a formula (linear regression), some by counting (naive Bayes), and some by **searching** (logistic regression).
- Fitting means finding the coefficients with the lowest **loss**: squared error for linear regression, and the average negative log-likelihood for logistic regression.
- The **slope** of the loss says which way is downhill (its sign) and how steep (its size).
- **Gradient descent** repeats one update: new value = old value $-$ (learning rate) $\times$ (slope). With several parameters, each one uses its own partial derivative, and together they make up the **gradient** $\nabla L$.
- The **learning rate** $\eta$ matters: too small is slow, too big overshoots or blows up. Stop when the loss barely changes.
- A **contour plot** is the loss surface seen from above. Descent steps cross the contour lines at right angles, which makes them **zigzag** in the narrow valleys that unscaled features create. Scale your features.
- Linear and logistic regression losses are single bowls, so gradient descent finds the best fit.

**What's next.** In Lab 3 (released Friday), you'll write gradient descent yourself: first for one parameter, then for two, and you'll choose a learning rate that works. In Week 9, gradient descent comes back to fit neural networks, whose losses aren't single bowls.

# Learning Check Practice

The learning check covers the ideas below. Try each question on paper before you open its hint or answer. Each question links back to the demo it depends on.

### Question 1: Formula, counting, or searching <a id="practice-q1"></a>

For each model, say how its coefficients are found: by a **formula**, by **counting**, or by **searching**.

(a) Linear regression &emsp; (b) Naive Bayes &emsp; (c) Logistic regression

Then explain in a sentence why logistic regression is the one that needs gradient descent. (See [Demo 4](#demo-4).)

<details>
<summary>Hint</summary>

Look at the table in [Demo 4](#demo-4). For each model, ask: can we write the best coefficients down directly from the data?

</details>

<details>
<summary>Answer</summary>

(a) **Formula.** Least squares gives the best coefficients in one calculation (it is also *possible* to find them by searching, using gradient descent, as we explore in the top of this notebook).

(b) **Counting.** The priors and the class-by-class estimates come from counts and averages in each class.

(c) **Searching.**

Logistic regression has no formula for its best coefficients, so we compute the loss and its gradient (slopes) at some chosen starting point (usually a random one) in parameter space. So we "seed" gradient descent somewhere and repeatedly step downhill until the gradient is nearly level (near the bottom of the minimum loss). That's gradient descent.

</details>

### Question 2: Which way is downhill? <a id="practice-q2"></a>

The demo below shows a loss $L(\beta_0)$ that depends on one parameter, $\beta_0$. It starts at a point where the tangent line has slope $-3$.

(a) Is the point depicted (the one drawn on the curve) to the left or to the right of the minimum?

(b) To move downhill, should $\beta_0$ increase or decrease?

(c) The update rule $\beta_0^{\text{new}} = \beta_0 - \eta\, L'(\beta_0)$ has a minus sign. What would gradient descent do if we changed that to a plus?

Try to answer from the slope alone before you look closely at the curve.

In [ ]:
from ch_4_demos_helpers import tangent_slope_demo

tangent_slope_demo(start=2.5, var='β₀', func='L', show_minimum=False, size_percent=85)

<details>
<summary>Hint</summary>

A negative slope means the curve falls as you move to the right. Drag the slider in the demo above and watch which points give a negative slope in the readout.

</details>

<details>
<summary>Answer</summary>

(a) **Left.** The curve is falling there, and it keeps falling until the minimum.

(b) **Increase.** The curve falls to the right, so the "downhill" direction is to the right (increasing $\beta_0$).

(c) **It would walk uphill.** Without the minus sign, a positive slope would move $\beta_0$ to the right and a negative slope would move it to the left, so every step would go the way the curve rises. The minus sign is what makes the step go *opposite* to the uphill direction.

</details>

### Question 3: One gradient descent step by hand <a id="practice-q3"></a>

Given a loss curve and its derivative:

$$
L(\beta_0) = (\beta_0 - 4)^2 + 2, \qquad L'(\beta_0) = 2(\beta_0 - 4)
$$

Start at $\beta_0 = 6$ with learning rate $\eta = 0.25$.

(a) Compute $L(6)$, the loss at the starting point.

(b) Compute the slope $L'(6)$.

(c) Compute the step $\Delta \beta_0 = -\eta \, L'(\beta_0)$. Does $\beta_0$ move left or right?

(d) Compute the new $\beta_0$, using $\beta_0^{\text{new}} = \beta_0 + \Delta \beta_0$.

(e) Compute $L(\beta_0^{\text{new}})$. Did the step go downhill?

<details>
<summary>Hint</summary>

Each part uses the answer to the part before it. In [Demo 3](#demo-3), the box above the plot works through the same arithmetic for each step it takes (with $x$ in place of $\beta_0$).

</details>

<details>
<summary>Answer</summary>

(a) $L(6) = (6 - 4)^2 + 2 = 4 + 2 = 6$

(b) $L'(6) = 2(6 - 4) = 4$. The slope is positive, so downhill is to the left.

(c) $\Delta \beta_0 = -0.25 \times 4 = -1$. $\beta_0$ moves **left** by 1.

(d) $\beta_0^{\text{new}} = 6 + (-1) = 5$

(e) $L(5) = (5 - 4)^2 + 2 = 1 + 2 = 3$. **Yes**: the loss dropped from 6 to 3.

The step also covered half the distance to the minimum, from 2 away to 1 away, just as Demo 3's $\eta = 0.25$ steps did.

</details>

### Question 4: Fixed steps versus scaled steps <a id="practice-q4"></a>

Treat the curves in Demos 2 and 3 as a loss $L(\beta_0)$, with $\beta_0$ on the horizontal axis. In [Demo 2](#demo-2), fixed steps of 0.25 starting from $\beta_0 = 6.3$ end up bouncing between 4.05 and 3.8 forever. In [Demo 3](#demo-3), steps of $\Delta \beta_0 = -0.25 \, L'(\beta_0)$ settle in toward the minimum at $\beta_0 = 4$.

What's different about the second rule that lets it settle? Use the slope in your answer.

<details>
<summary>Hint</summary>

Watch the slope column in the table of each demo as the point gets close to 4. What happens to the size of each step?

</details>

<details>
<summary>Answer</summary>

In the second rule, the **step size is proportional to the slope**, and the slope shrinks as the point nears the bottom (it's 0 at the bottom). So the steps shrink on their own and get smaller than the remaining distance, and the point closes in on 4.

A fixed step is always 0.25 long, no matter how close we are. Once the point is within 0.25 of the bottom, every step jumps over it, so it can only bounce back and forth.

</details>

### Question 5: Reading the loss plot <a id="practice-q5"></a>

Here are three runs of gradient descent, described by their loss-versus-step plots, like the right-hand plot in [Demo 7](#demo-7). Match each run to a learning rate that is **too small**, **about right**, or **too big**.

(a) The loss goes up on every step.

(b) The loss drops quickly, then levels off and stays flat.

(c) The loss falls smoothly, but slowly. After many steps, it's still falling.

Section 7 suggests stopping when the loss changes by less than 0.001 from one step to the next. Which of the three runs would stop by that rule soonest?

<details>
<summary>Hint</summary>

In [Demo 7](#demo-7), click the presets and watch the right-hand plot. A loss that ever goes up is a warning sign.

</details>

<details>
<summary>Answer</summary>

(a) **Too big.** Each step overshoots the bottom by more than the last, so the loss climbs.

(b) **About right.** Big early progress, then the loss stops changing once it's at the bottom.

(c) **Too small.** Every step goes downhill, but the steps are tiny.

Run **(b)** stops soonest: once the loss levels off, it changes by less than 0.001 per step. Run (c) keeps changing for a long time. Run (a) never meets the rule, which is why we also set a maximum number of steps.

</details>

### Question 6: Reading the gradient <a id="practice-q6"></a>

At some point $(\beta_0, \beta_1)$, the gradient of the loss is $\nabla L = (6, -2)$. (See [Demo 8b](#demo-8b).)

(a) To decrease the loss, should $\beta_0$ increase or decrease? Should $\beta_1$ increase or decrease?

(b) Which changes the loss more, "move $\beta_0$ only" or "move $\beta_1$ only"?

(c) With $\eta = 0.5$, how much does one step change each parameter?

<details>
<summary>Hint</summary>

The gradient is the list of the two slopes: $\nabla L = \left(\frac{\partial L}{\partial \beta_0}, \frac{\partial L}{\partial \beta_1}\right)$. Treat each one like the single slope in Question 3. In [Demo 8b](#demo-8b), compare the two small plots with the readout.

</details>

<details>
<summary>Answer</summary>

(a) Downhill is $-\nabla L = (-6, 2)$. So $\beta_0$ should *decrease* and $\beta_1$ should *increase*.

(b) **"Moving $\beta_0$ only."** Its slope is 6, versus 2 for $\beta_1$ (ignoring signs, the sizes are what matter for steepness).

(c) The step is $-\eta \, \nabla L = -0.5 \times (6, -2) = (-3, 1)$, so $\beta_0$ **decreases by 3** and $\beta_1$ **increases by 1**.

</details>

### Question 7: Why scale features? <a id="practice-q7"></a>

In [Demo 9b](#demo-9b), at 50% of the largest stable learning rate, standardized TV reaches the bottom in 1 step and unscaled TV takes 52. The data and the best line are the same in both versions.

(a) What about the unscaled version's contour plot causes the extra steps? 

(b) What should you do about it before running gradient descent?

<details>
<summary>Hint</summary>

Compare the shapes of the two contour plots in [Demo 9b](#demo-9b). Then go back to [Demo 9a](#demo-9a): from a point on the wall of the valley, where does the downhill arrow point?

</details>

<details>
<summary>Answer</summary>

(a) Unscaled TV turns the round bowl into a **long, narrow, tilted valley**. Each step goes straight across the contour lines, and in a narrow valley that points mostly *across* the valley, not along it toward the minimum. So the steps zigzag from wall to wall. Once the path reaches the valley floor, the slopes are small, so it crawls toward the minimum.

The learning rate makes this worse. One $\eta$ has to work in both directions. An $\eta$ small enough not to overshoot across the steep valley is too small to make quick progress along the gently sloped valley floor.

(b) **Scale the features** (for example, standardize them) before running gradient descent. Then the contours are close to circles, and the downhill direction points at the minimum.

</details>

## Stretch (optional)

### Stretch Question: Where does the loss surface come from?

The surface in [Demo 8a](#demo-8a) wasn't drawn by hand. It was computed from the loss function and the data. For linear regression with one feature, the loss is the mean squared error from Section 5:

$$
L(\beta_0, \beta_1) = \frac{1}{n} \sum_{i=1}^{n} \big(y_i - \beta_0 - \beta_1 x_i\big)^2
$$

Here is a tiny training set with $n = 2$ examples:

| $i$ | $x_i$ | $y_i$ |
| --- | --- | --- |
| 1 | 1 | 2 |
| 2 | 2 | 4 |

(a) Compute $L(0, 2)$, the loss for the line $\hat{y} = 0 + 2x$.

(b) Compute $L(0, 1)$, the loss for the line $\hat{y} = 0 + 1x$.

(c) Each answer is the height of the loss surface at one point $(\beta_0, \beta_1)$. What does that point stand for? In the formula, which symbols were fixed numbers, and which ones did you choose?

(d) Suppose the second example is $(2, 6)$ instead of $(2, 4)$. Recompute $L(0, 2)$ and $L(0, 1)$. Did the loss formula change? Did the surface change? Is $(0, 2)$ still the bottom of the bowl?

(e) What two things do you need in order to compute the loss surface? The `Default` data has 10,000 customers. Why doesn't gradient descent just compute the whole surface and pick its lowest point?

<details>
<summary>Hint</summary>

- For each example, compute the prediction $\beta_0 + \beta_1 x_i$, subtract it from $y_i$, and square. Then average the squares.
- For (e), count the work: how many examples go into a single height? How many heights are there?
- Recall the goal from Section 1: move toward the bottom using *only* the slope where you are now.

</details>

<details>
<summary>Answer</summary>

(a) The line $\hat{y} = 2x$ predicts 2 and 4, so both errors are 0:

$$
L(0, 2) = \tfrac{1}{2}\big[(2 - 2)^2 + (4 - 4)^2\big] = 0
$$

(b) The line $\hat{y} = x$ predicts 1 and 2, so the errors are 1 and 2:

$$
L(0, 1) = \tfrac{1}{2}\big[(2 - 1)^2 + (4 - 2)^2\big] = \tfrac{1}{2}(1 + 4) = 2.5
$$

(c) Each point $(\beta_0, \beta_1)$ in parameter space is **one model**: here, one line. Its height is how badly that line fits *this* training data. The $x_i$ and $y_i$ were **fixed numbers** from the table. The **variables** were $\beta_0$ and $\beta_1$, which we chose. That's the reverse of how we usually use a model, where the coefficients are fixed and $x$ varies.

(d) With $(2, 6)$ as the second example:

$$
L(0, 2) = \tfrac{1}{2}\big[(2 - 2)^2 + (6 - 4)^2\big] = 2, \qquad
L(0, 1) = \tfrac{1}{2}\big[(2 - 1)^2 + (6 - 2)^2\big] = \tfrac{1}{2}(1 + 16) = 8.5
$$

The loss formula didn't change, but both heights did, so **the surface changed**. $(0, 2)$ is no longer the bottom: its loss went from 0 to 2. The bottom moved to a line that fits the new data better. (The line through $(1, 2)$ and $(2, 6)$, $\hat{y} = -2 + 4x$, has loss 0.) The best model belongs to the data it was fit to.

(e) You need **the loss function** (which includes the model's prediction formula, $\beta_0 + \beta_1 x_i$) and **the training data**. Change either one and you get a different surface.

Gradient descent doesn't compute the whole surface because every single height costs a pass through all the training examples: 10,000 predictions and errors for `Default`. A fine grid over just two parameters could be 10,000 points, which means 100 million predictions. With more parameters, the number of grid points multiplies with each one. Gradient descent only computes the loss and the slopes at the points it actually visits, which is a few thousand at most (in this example). That's why Section 1 set the goal of finding the bottom using only the slope where you are.

</details>